In [1]:
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

from shapely.geometry import Point
from rasterio.mask import mask

In [2]:
raster_path = "../data/raw/VNL_v21_npp_2013_global_vcmcfg_c202205302300.average_masked.dat.tif"

with rasterio.open(raster_path) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Count:", src.count)
    print("Dtype:", src.dtypes)
    print("Nodata:", src.nodata)

CRS: EPSG:4326
Bounds: BoundingBox(left=-180.00208333335, bottom=-65.00208445335001, right=180.00208621335, top=75.00208333335)
Width: 86401
Height: 33601
Count: 1
Dtype: ('float32',)
Nodata: None


In [3]:
cities = pd.DataFrame([
    {"city": "Madrid", "country": "Spain", "lat": 40.4168, "lon": -3.7038},
    {"city": "Paris", "country": "France", "lat": 48.8566, "lon": 2.3522},
    {"city": "Berlin", "country": "Germany", "lat": 52.5200, "lon": 13.4050},
])

cities

,city,country,lat,lon
0,Madrid,Spain,40.4168,-3.7038
1,Paris,France,48.8566,2.3522
2,Berlin,Germany,52.5200,13.4050


In [4]:
city_row = cities[cities["city"] == "Madrid"].iloc[0]

gdf = gpd.GeoDataFrame(
    [city_row],
    geometry=[Point(city_row["lon"], city_row["lat"])],
    crs="EPSG:4326"
)

# Reproject to a metric CRS for buffering in meters
gdf_m = gdf.to_crs("EPSG:3035")

# 25 km buffer
gdf_m["geometry"] = gdf_m.buffer(25000)

# Convert back to lat/lon to match the raster CRS
gdf_buffer = gdf_m.to_crs("EPSG:4326")

gdf_buffer

,city,country,lat,lon,geometry
0,Madrid,Spain,40.4168,-3.7038,"POLYGON ((-3.4139 40.45709, -3.41065 40.43512,..."


In [5]:
with rasterio.open(raster_path) as src:
    out_image, out_transform = mask(src, gdf_buffer.geometry, crop=True)
    data = out_image[0]

print("Masked array shape:", data.shape)
print("Min value:", np.nanmin(data))
print("Max value:", np.nanmax(data))

Masked array shape: (109, 142)
Min value: 0.0
Max value: 233.3496


In [6]:
# Flatten the array
values = data.flatten()

# Remove NaNs
values = values[~np.isnan(values)]

# Remove extreme negative values if present
values = values[values >= 0]

print("Number of valid pixels:", len(values))
print("Mean brightness:", float(values.mean()))
print("Median brightness:", float(np.median(values)))
print("90th percentile brightness:", float(np.percentile(values, 90)))

Number of valid pixels: 15478
Mean brightness: 24.2916259765625
Median brightness: 4.728999614715576
90th percentile brightness: 86.87882995605469


In [7]:
result = pd.DataFrame([{
    "city": "Madrid",
    "year": 2013,
    "mean_brightness": float(values.mean()),
    "median_brightness": float(np.median(values)),
    "p90_brightness": float(np.percentile(values, 90)),
    "pixel_count": int(len(values))
}])

result

,city,year,mean_brightness,median_brightness,p90_brightness,pixel_count
0,Madrid,2013,24.291626,4.729,86.87883,15478


In [8]:
result.to_csv("../data/interim/madrid_viirs_2013_sample.csv", index=False)